In [ ]:
%%capture
import os
from pathlib import Path

import pandas as pd
from dj_notebook import activate

env_file = os.environ["META_ENV"]
reports_folder = Path(os.environ["META_REPORTS_FOLDER"])
analysis_folder = Path(os.environ["META_ANALYSIS_FOLDER"])
pharmacy_folder = Path(os.environ["META_PHARMACY_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option("future.no_silent_downcasting", True)

In [ ]:
# calculate the number of appointments from 1 jan 2026 to study.
# Assume all subjects agree to extended followup

In [ ]:
from zoneinfo import ZoneInfo
from edc_appointment.analytics import get_appointment_df
from edc_appointment.constants import SCHEDULED_APPT, ONTIME_APPT, NEW_APPT
from datetime import datetime

# linter annoyingly removes these constants because they
# only appear in df.query() string
SCHEDULED_APPT
ONTIME_APPT
NEW_APPT

In [ ]:
end_of_trial_date = datetime(2026, 6, 1, tzinfo=ZoneInfo("UTC"))

In [ ]:
df_appointments = get_appointment_df()
df_appointments["site_id"] = df_appointments.site_id.astype(str)

In [ ]:
# 1480 not attended
df_appt_1480 = df_appointments.query(
    "appt_datetime<=@end_of_trial_date and "
    "appt_reason==@SCHEDULED_APPT and "
    "appt_timing==@ONTIME_APPT and "
    "schedule_name=='schedule' and "
    "visit_code>=1480.0"
)

In [ ]:
# 1360 no 1480 not attended
df_appt_all = df_appointments.query(
    "appt_reason==@SCHEDULED_APPT and "
    "appt_timing==@ONTIME_APPT and "
    "appt_status.isin([@NEW_APPT]) and "
    "schedule_name=='schedule' and "
    "visit_code>=1360.0"
)[["subject_identifier", "visit_code", "appt_datetime", "appt_status"]].sort_values([
    "subject_identifier", "visit_code"])

In [ ]:
# reomve all subjects who have a 1480; that is, reconsented to extended followup
df = df_appt_all[(~df_appt_all["subject_identifier"].isin(
    df_appt_1480["subject_identifier"]))].sort_values([
    "subject_identifier", "visit_code"])

# these are the subjects that did not reconsent
df_not_consented = df[~df.duplicated('subject_identifier', keep=False)]

In [ ]:
# next create fake appts as if these subjects have extended

In [ ]:
# fetch to view 1360s. Everyone on schedule has a 1360 regardless of
# re-consenting for extended followup
df_1360 = df_not_consented[
    (df_not_consented[
         "appt_datetime"] >= datetime(2026, 1, 1, tzinfo=ZoneInfo("UTC"))) & (
            df_not_consented["appt_datetime"] <= end_of_trial_date)
    ]
df_1360.sort_values("subject_identifier")

In [ ]:
# create 1390s to add ...
df_1390 = df_1360.copy()
df_1390["appt_datetime"] = df_1390["appt_datetime"] + pd.Timedelta(weeks=12)
df_1390["visit_code"] = 1390.0
df_1390 = df_1390.sort_values("subject_identifier")

In [ ]:
# create 1420s to add ...
df_1420 = df_1390.copy()
df_1420["appt_datetime"] = df_1420["appt_datetime"] + pd.Timedelta(weeks=12)
df_1420["visit_code"] = 1420.0
df_1420 = df_1420.sort_values("subject_identifier")

In [ ]:
# create 1450s to add ...
df_1450 = df_1420.copy()
df_1450["appt_datetime"] = df_1450["appt_datetime"] + pd.Timedelta(weeks=12)
df_1450["visit_code"] = 1450.0
df_1450 = df_1450.sort_values("subject_identifier")

In [ ]:
# create 1480s to add ...
df_1480 = df_1450.copy()
df_1480["appt_datetime"] = df_1480["appt_datetime"] + pd.Timedelta(weeks=12)
df_1480["visit_code"] = 1480.0
df_1480 = df_1480.sort_values("subject_identifier")

In [ ]:
# concat the "fake" appts into a single dataframe
df_fake_appts = pd.concat([df_1390, df_1420, df_1450, df_1480]).sort_values([
    "subject_identifier",
    "visit_code"])


In [ ]:
# get all "real" appts. filter for NEW_APPT only
df_real_appts = df_appointments[(df_appointments["appt_status"] == NEW_APPT)][["subject_identifier", "visit_code", "appt_datetime", "appt_status"]]


In [ ]:
# concat and date filter real and fake dataframes
dte1 = datetime(2026,1,1,tzinfo=ZoneInfo("UTC"))
dte2 = datetime(2026,6,1,tzinfo=ZoneInfo("UTC"))
df_all = pd.concat([df_real_appts,df_fake_appts], ignore_index=True).query("appt_datetime>=@dte1 and appt_datetime<@dte2")
df_all["appt_datetime"] = df_all["appt_datetime"].dt.tz_localize(None)

In [ ]:
# export this to file as a summary my month
df_all.groupby([df_all['appt_datetime'].dt.to_period('M'), 'visit_code']).size().to_frame().reset_index()

In [ ]:
# export this to file as a summary by visit code
df_all.groupby(['visit_code']).size().to_frame().reset_index()